# Sportmonks Datenakquise — Matchstatistiken Super League

Dieses Notebook ergänzt die API-Football Basisdaten um **detaillierte Spielstatistiken**
aus der Sportmonks Football API v3.

**Was Sportmonks zusätzlich liefert (pro Spiel, pro Team):**
- 🔵 Ballbesitz (%)
- 🎯 Schüsse gesamt / aufs Tor / daneben / geblockt
- 🟨 Gelbe & Rote Karten
- 🚩 Eckbälle, Fouls, Abseits
- 🧤 Saves (Torwart-Paraden)
- ⚡ Angriffe / Gefährliche Angriffe
- 🔁 Pässe gesamt / erfolgreich

**Voraussetzung:** `.env` im Projekt-Root mit:
```
API_SPORTMONKS_KEY="dein_key"
API_SPORTMONKS_URL="https://api.sportmonks.com/v3/football"
```

**Strategie:** Zuerst Liga-ID und Saison-ID ermitteln, dann alle Fixtures mit
Statistiken laden und in ein breites CSV-Format pivotieren.

## 1. Setup & Imports

In [ ]:
import os
import time
import requests
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

# .env laden (Projekt-Root)
env_path = Path("__file__").resolve().parent.parent / ".env"
load_dotenv(dotenv_path=env_path)

API_KEY  = os.environ["API_SPORTMONKS_KEY"]
BASE_URL = os.environ.get("API_SPORTMONKS_URL", "https://api.sportmonks.com/v3/football")

# Validierung: Warnung falls die Docs-URL eingetragen wurde
if "docs.sportmonks" in BASE_URL:
    raise ValueError(
        "API_SPORTMONKS_URL zeigt auf die Doku, nicht die API!\n"
        "Bitte in .env setzen: API_SPORTMONKS_URL=https://api.sportmonks.com/v3/football"
    )

HEADERS = {"Authorization": API_KEY}

RAW_DIR = Path("raw")
RAW_DIR.mkdir(exist_ok=True)

print(f"Base URL : {BASE_URL}")
print(f"API Key  : {'✓ geladen' if API_KEY else '✗ FEHLT – .env prüfen!'}")
print(f"Output   : {RAW_DIR.resolve()}")

## 2. Hilfsfunktionen

In [ ]:
def api_get(endpoint: str, params: dict = None) -> dict:
    """Einzelner GET-Request. Gibt das vollständige JSON zurück."""
    url = f"{BASE_URL}/{endpoint.lstrip('/')}"
    resp = requests.get(url, headers=HEADERS, params=params or {})
    resp.raise_for_status()
    return resp.json()


def api_get_all(endpoint: str, params: dict = None) -> list:
    """Paginierter GET-Request: lädt alle Seiten und gibt eine flache Liste zurück."""
    params = params or {}
    params["per_page"] = 50
    all_data = []
    page = 1

    while True:
        params["page"] = page
        data = api_get(endpoint, params)
        items = data.get("data", [])
        all_data.extend(items)

        pagination = data.get("pagination", {})
        has_more = pagination.get("has_more", False)
        rate_remaining = data.get("rate_limit", {}).get("remaining", "?")

        print(f"  Seite {page} | {len(items)} Einträge | Rate-Limit verbleibend: {rate_remaining}")

        if not has_more:
            break
        page += 1
        time.sleep(0.3)

    return all_data

## 3. Statistik-Typen (type_id → Bezeichnung)

Sportmonks kodiert alle Statistiken als `type_id` (z.B. type_id 45 = Ballbesitz).
Wir laden alle Typen einmalig und erstellen ein Mapping-Dict.

In [ ]:
print("Lade Statistik-Typen...")
types_data = api_get_all("types/entity/fixture")

# Mapping: type_id → sauberer Spaltenname
TYPE_MAP = {}
for t in types_data:
    type_id   = t["id"]
    type_name = t.get("name", f"type_{type_id}")
    # Zu sauberem Spaltennamen konvertieren
    col_name = (
        type_name.lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("-", "_")
        .replace("(", "")
        .replace(")", "")
        .replace("%", "pct")
    )
    TYPE_MAP[type_id] = col_name

print(f"\n{len(TYPE_MAP)} Statistik-Typen geladen.")
print("\nBeispiele:")
for tid, name in sorted(TYPE_MAP.items())[:20]:
    print(f"  {tid:>4}: {name}")

## 4. Swiss Super League finden

In [ ]:
print("Suche Swiss Super League...")
leagues_data = api_get_all(
    "leagues",
    params={"filters": "countryCode:CH", "include": "currentSeason"}
)

df_leagues = pd.DataFrame([
    {
        "league_id":   lg["id"],
        "name":        lg["name"],
        "short_code":  lg.get("short_code", ""),
        "type":        lg.get("type", ""),
        "season_id":   lg.get("currentSeason", {}).get("id") if lg.get("currentSeason") else None,
        "season_name": lg.get("currentSeason", {}).get("name") if lg.get("currentSeason") else None,
    }
    for lg in leagues_data
])

print("\nSchweizer Ligen gefunden:")
print(df_leagues[["league_id", "name", "type", "season_id", "season_name"]].to_string(index=False))

In [ ]:
# Super League manuell aus der obigen Tabelle auswählen
# Passe LEAGUE_ID und SEASON_ID anhand der Ausgabe oben an!

super_league_row = df_leagues[df_leagues["name"].str.contains("Super League", case=False, na=False)]

if super_league_row.empty:
    print("❌ 'Super League' nicht gefunden — bitte LEAGUE_ID und SEASON_ID manuell setzen.")
    LEAGUE_ID = None
    SEASON_ID = None
else:
    LEAGUE_ID = int(super_league_row.iloc[0]["league_id"])
    SEASON_ID = int(super_league_row.iloc[0]["season_id"])
    print(f"✅ Swiss Super League gefunden!")
    print(f"   League ID : {LEAGUE_ID}")
    print(f"   Season ID : {SEASON_ID} ({super_league_row.iloc[0]['season_name']})")

## 5. Fixtures mit Statistiken laden

Wir laden alle abgeschlossenen Spiele (`status: FT`) der aktuellen Saison
inklusive Statistiken, Scores und Teilnehmer.

In [ ]:
assert LEAGUE_ID and SEASON_ID, "LEAGUE_ID / SEASON_ID nicht gesetzt — Zelle 4 prüfen!"

print(f"Lade Fixtures für Season ID {SEASON_ID}...")
fixtures_raw = api_get_all(
    f"fixtures/seasons/{SEASON_ID}",
    params={
        "include": "statistics;participants;scores;state",
    }
)

print(f"\n{len(fixtures_raw)} Fixtures geladen (inkl. geplanter Spiele).")

## 6. Fixtures flachklopfen & Statistiken pivotieren

In [ ]:
def extract_participants(participants: list) -> tuple[dict, dict]:
    """Gibt (home_team, away_team) als Dicts zurück."""
    home = {"team_id": None, "team_name": ""}
    away = {"team_id": None, "team_name": ""}
    for p in participants or []:
        meta = p.get("meta", {})
        location = meta.get("location", "")
        if location == "home":
            home = {"team_id": p["id"], "team_name": p.get("name", "")}
        elif location == "away":
            away = {"team_id": p["id"], "team_name": p.get("name", "")}
    return home, away


def extract_score(scores: list, period: str = "FULL TIME") -> tuple[int | None, int | None]:
    """Gibt (home_score, away_score) für den angegebenen Abschnitt zurück."""
    for sc in scores or []:
        description = sc.get("description", "").upper()
        if description == period.upper():
            goals = sc.get("score", {})
            return goals.get("goals"), goals.get("participant")  # home/away encoded
    return None, None


def extract_stats(statistics: list, type_map: dict) -> tuple[dict, dict]:
    """Pivotiert die Statistik-Liste in zwei flache Dicts (home_stats, away_stats)."""
    home_stats: dict = {}
    away_stats: dict = {}
    for stat in statistics or []:
        type_id  = stat.get("type_id")
        location = stat.get("location", "")
        value    = stat.get("data", {}).get("value")
        col_name = type_map.get(type_id, f"stat_{type_id}")
        if location == "home":
            home_stats[col_name] = value
        elif location == "away":
            away_stats[col_name] = value
    return home_stats, away_stats


print("Verarbeite Fixtures...")
fixture_rows_home = []
fixture_rows_away = []
skipped = 0

for fx in fixtures_raw:
    state = fx.get("state", {}) or {}
    status = state.get("short_name", "") if isinstance(state, dict) else ""

    # Nur abgeschlossene Spiele
    if status not in ("FT", "AET", "PEN"):
        skipped += 1
        continue

    home_team, away_team = extract_participants(fx.get("participants", []))
    home_stats, away_stats = extract_stats(fx.get("statistics", []), TYPE_MAP)

    base = {
        "fixture_id":    fx["id"],
        "date":          fx.get("starting_at", ""),
        "round":         fx.get("round_id", ""),
        "status":        status,
        "home_team_id":  home_team["team_id"],
        "home_team":     home_team["team_name"],
        "away_team_id":  away_team["team_id"],
        "away_team":     away_team["team_name"],
    }

    # Für Scores: aus scores-Array
    for sc in fx.get("scores", []) or []:
        desc = sc.get("description", "").upper()
        score = sc.get("score", {})
        if desc == "CURRENT":
            base["goals_home"] = score.get("goals")   if score.get("participant") == "home" else None
            base["goals_away"] = score.get("goals")   if score.get("participant") == "away" else None

    # Home-Zeile
    home_row = {**base, "perspective": "home", "team_id": home_team["team_id"], "team_name": home_team["team_name"]}
    home_row.update(home_stats)
    fixture_rows_home.append(home_row)

    # Away-Zeile
    away_row = {**base, "perspective": "away", "team_id": away_team["team_id"], "team_name": away_team["team_name"]}
    away_row.update(away_stats)
    fixture_rows_away.append(away_row)

df_fixture_stats = pd.concat([
    pd.DataFrame(fixture_rows_home),
    pd.DataFrame(fixture_rows_away),
], ignore_index=True).sort_values(["fixture_id", "perspective"])

df_fixture_stats.to_csv(RAW_DIR / "fixture_statistics.csv", index=False)

n_fixtures = len(df_fixture_stats) // 2
print(f"\n✅ fixture_statistics.csv gespeichert")
print(f"   {n_fixtures} Spiele | {skipped} noch ausstehend/geplant")
print(f"   {len(df_fixture_stats.columns)} Spalten")
print(f"\nVerfügbare Statistikspalten:")
stat_cols = [c for c in df_fixture_stats.columns if c not in [
    "fixture_id", "date", "round", "status",
    "home_team_id", "home_team", "away_team_id", "away_team",
    "perspective", "team_id", "team_name", "goals_home", "goals_away"
]]
print("  ", ", ".join(stat_cols))

## 7. Team-Aggregat: Saisondurchschnitte pro Team

Aus den Spiel-für-Spiel-Daten berechnen wir Saisondurchschnitte pro Team —
das sind die Daten für Radar Charts und Scatter Plots.

In [ ]:
# Nur numerische Statistikspalten aggregieren
exclude_from_agg = {
    "fixture_id", "date", "round", "status",
    "home_team_id", "home_team", "away_team_id", "away_team",
    "perspective", "goals_home", "goals_away"
}

numeric_stat_cols = [
    c for c in df_fixture_stats.columns
    if c not in exclude_from_agg
    and c not in {"team_id", "team_name"}
    and pd.api.types.is_numeric_dtype(df_fixture_stats[c])
]

agg_dict = {col: "mean" for col in numeric_stat_cols}
agg_dict["fixture_id"] = "count"  # = Anzahl Spiele

df_team_agg = (
    df_fixture_stats
    .groupby(["team_id", "team_name"])
    .agg(agg_dict)
    .rename(columns={"fixture_id": "games_played"})
    .reset_index()
)

# Spaltennamen mit _avg-Suffix versehen (ausser games_played)
rename_map = {
    c: f"{c}_avg" for c in numeric_stat_cols
}
df_team_agg = df_team_agg.rename(columns=rename_map)

df_team_agg.to_csv(RAW_DIR / "team_stats_sportmonks.csv", index=False)

print(f"✅ team_stats_sportmonks.csv gespeichert")
print(f"   {len(df_team_agg)} Teams | {len(df_team_agg.columns)} Spalten")

# Vorschau: die interessantesten Spalten
preview_cols = ["team_name", "games_played"] + [
    c for c in df_team_agg.columns
    if any(kw in c for kw in ["possession", "shot", "ball", "corner", "foul", "card", "attack"])
][:10]
print()
df_team_agg[preview_cols].sort_values("games_played", ascending=False)

## 8. Saisonverlauf FC Thun — Punkte kumuliert

Aus den Fixtures rekonstruieren wir den Punkteverlauf über die Spieltage.
Das ist die Grundlage für das Liniendiagramm im Blog.

In [ ]:
# API-Football standings einlesen (falls vorhanden)
standings_path = RAW_DIR / "standings.csv"
if standings_path.exists():
    df_standings = pd.read_csv(standings_path)
    print("Standings aus API-Football geladen:")
    print(df_standings[["rank", "team_name", "points", "played"]].to_string(index=False))
else:
    print("ℹ️  standings.csv nicht gefunden — zuerst fetch_data.ipynb (API-Football) ausführen.")

# Saisonverlauf aus Fixtures berechnen
# Wir verwenden die home-Perspektive um Duplikate zu vermeiden
df_home = df_fixture_stats[df_fixture_stats["perspective"] == "home"].copy()
df_away = df_fixture_stats[df_fixture_stats["perspective"] == "away"].copy()

# Punkte pro Spiel: 3 Sieg, 1 Unentschieden, 0 Niederlage
def compute_points(goals_scored, goals_conceded):
    if pd.isna(goals_scored) or pd.isna(goals_conceded):
        return None
    if goals_scored > goals_conceded:
        return 3
    elif goals_scored == goals_conceded:
        return 1
    else:
        return 0

# Für jedes Spiel: Punkte berechnen
timeline_rows = []
for _, row in df_home.iterrows():
    # Home team
    pts_home = compute_points(row.get("goals_home"), row.get("goals_away"))
    timeline_rows.append({
        "fixture_id": row["fixture_id"],
        "date":       row["date"],
        "round":      row["round"],
        "team_id":    row["home_team_id"],
        "team_name":  row["home_team"],
        "goals_for":     row.get("goals_home"),
        "goals_against": row.get("goals_away"),
        "points":     pts_home,
    })

for _, row in df_away.iterrows():
    # Away team
    pts_away = compute_points(row.get("goals_away"), row.get("goals_home"))
    timeline_rows.append({
        "fixture_id": row["fixture_id"],
        "date":       row["date"],
        "round":      row["round"],
        "team_id":    row["away_team_id"],
        "team_name":  row["away_team"],
        "goals_for":     row.get("goals_away"),
        "goals_against": row.get("goals_home"),
        "points":     pts_away,
    })

df_timeline = pd.DataFrame(timeline_rows)
df_timeline["date"] = pd.to_datetime(df_timeline["date"])
df_timeline = df_timeline.sort_values(["team_name", "date"]).reset_index(drop=True)

# Kumulierte Punkte
df_timeline["points_cumulative"] = df_timeline.groupby("team_name")["points"].cumsum()
df_timeline["match_nr"] = df_timeline.groupby("team_name").cumcount() + 1

df_timeline.to_csv(RAW_DIR / "season_timeline.csv", index=False)

print(f"\n✅ season_timeline.csv gespeichert")
print(f"   {len(df_timeline)} Zeilen (1 pro Team pro Spiel)")

# Vorschau FC Thun
thun = df_timeline[df_timeline["team_name"].str.contains("Thun", na=False)]
if not thun.empty:
    print(f"\nFC Thun Saisonverlauf (letzte 5 Spiele):")
    print(thun[["match_nr", "date", "goals_for", "goals_against", "points", "points_cumulative"]].tail(5).to_string(index=False))
else:
    print("\nℹ️  'Thun' nicht in den Fixtures gefunden — Teamnamen prüfen.")

## 9. Übersicht aller gespeicherten Dateien

In [ ]:
print("Gespeicherte Dateien in data_acquisition/raw/:\n")
for csv_file in sorted(RAW_DIR.glob("*.csv")):
    df = pd.read_csv(csv_file)
    size_kb = csv_file.stat().st_size / 1024
    print(f"  {csv_file.name:<35} {len(df):>4} Zeilen × {len(df.columns):>3} Spalten   ({size_kb:.1f} KB)")

print("\n🏁 Sportmonks Datenakquise abgeschlossen.")
print("   Nächster Schritt: uv run python eda/generate-data-profile.py")